For this analysis we will be using data pulled from the cancer genome atlas. To access this data I used the USC Xena Browser to download clean data. The specific datasets I will be using are (downloaded February 20, 2026):

- TCGA-BRCA.clinical.tsv
    - contains sample type (disease vs healthy)
- TCGA-BRCA.star_counts.tsv
    - RNAseq data
- TCGA-BRCA.survival.tsv
- TCGA-BRCA.protein.tsv

Xena Browser
https://xenabrowser.net/datapages/?cohort=GDC%20TCGA%20Breast%20Cancer%20(BRCA)&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

PAM50 Subtypes (Direct Download Link)
https://api.gdc.cancer.gov/data/a2a299a9-6cad-4811-a74c-0bdfa61cbab2 

In [93]:
import pandas as pd
import os
import gzip
import shutil
from pathlib import Path

First, we need to decompress the data we downloaded. Below is a short script that will extract any compressed .gz files and delete the compressed file. Then, it prints the files that are currently in the data/raw/ directory.

In [94]:
raw_dir = Path("../data/raw")

# Get .gz files in data/raw/
for gz_file in sorted(raw_dir.glob("*.gz")):
    out_file = gz_file.with_suffix("")  # strip .gz

    # If already decompressed, skip
    if out_file.exists():
        print(f"Skipping {gz_file.name} (already decompressed)")
        continue
    
    # Decompress .gz file
    print(f"Decompressing {gz_file.name} ...", end=" ")
    with gzip.open(gz_file, "rb") as f_in, open(out_file, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    gz_file.unlink()
    print(f"done -> {out_file.name}")

# Print files in raw dir
print("\nFiles in raw directory:")
for f in sorted(raw_dir.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:45s} {size_mb:>8.1f} MB")


Files in raw directory:
  BRCA.547.PAM50.SigClust.Subtypes.txt               0.0 MB
  TCGA-BRCA.clinical.tsv                             1.7 MB
  TCGA-BRCA.protein.tsv                              4.3 MB
  TCGA-BRCA.star_counts.tsv                        719.8 MB
  TCGA-BRCA.survival.tsv                             0.0 MB


Now that the data is available in our data/raw/ directory, we can load it into pandas.

In [95]:
df_star = pd.read_csv('/Users/nick/projects/bioinformatics-project/data/raw/TCGA-BRCA.star_counts.tsv', sep='\t', index_col=0)
df_clinical = pd.read_csv('/Users/nick/projects/bioinformatics-project/data/raw/TCGA-BRCA.clinical.tsv', sep='\t')
df_survival = pd.read_csv('/Users/nick/projects/bioinformatics-project/data/raw/TCGA-BRCA.survival.tsv', sep='\t')
df_pam50 = pd.read_csv('/Users/nick/projects/bioinformatics-project/data/raw/BRCA.547.PAM50.SigClust.Subtypes.txt', sep='\t')
df_star.head()

,TCGA-D8-A146-01A,TCGA-AQ-A0Y5-01A,TCGA-C8-A274-01A,TCGA-BH-A0BD-01A,TCGA-B6-A1KC-01B,TCGA-AC-A62V-01A,TCGA-AO-A0J5-01A,TCGA-BH-A0B1-01A,TCGA-A2-A0YM-01A,TCGA-AO-A03N-01B,...,TCGA-E2-A1IG-01A,TCGA-E9-A1NA-01A,TCGA-D8-A1JP-01A,TCGA-AR-A252-01A,TCGA-D8-A1XL-01A,TCGA-BH-A0EI-01A,TCGA-E2-A1IO-01A,TCGA-E2-A15R-01A,TCGA-B6-A0IP-01A,TCGA-A1-A0SN-01A
Ensembl_ID,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003.15,11.737670,9.781360,13.122504,11.016808,11.000000,9.614710,11.092096,12.103616,12.417325,10.430453,...,10.747354,10.360847,10.675957,11.337064,11.697836,12.339850,12.131857,8.974415,13.326289,9.710806
ENSG00000000005.6,7.721099,3.321928,0.000000,6.686501,3.807355,4.087463,5.392317,3.000000,3.321928,0.000000,...,1.584963,2.000000,1.000000,8.164907,0.000000,2.807355,3.906891,3.700440,3.807355,1.000000
ENSG00000000419.13,11.042343,11.357552,11.506308,10.801708,11.074141,11.107217,10.318543,11.160502,11.072803,11.088126,...,10.429407,10.811375,10.835261,10.693487,11.922213,11.221587,10.913637,11.659550,10.962173,12.416270
ENSG00000000457.14,11.036860,10.754888,12.218260,11.190442,10.857981,8.539159,11.527966,10.635718,9.945444,9.499846,...,10.366322,10.558421,11.262682,10.751544,10.607330,10.163650,10.575539,11.929258,11.403012,10.870365
ENSG00000000460.17,9.131857,8.721099,10.973697,10.761551,9.550747,8.550747,9.177420,9.727920,10.504819,8.584963,...,8.661778,9.000000,9.794416,9.231221,9.782998,9.252665,8.632995,10.562242,10.060696,9.722808


In [96]:
# Check shape of dfs
print(df_star.shape, df_clinical.shape, df_survival.shape, df_pam50.shape)

# Print columns for the dataframes
print('df_star:', df_star.columns.tolist())
print('df_clinical:', df_clinical.columns.tolist())
print('df_survival:', df_survival.columns.tolist())
print('df_pam50:', df_pam50.columns.tolist())

(60660, 1226) (1255, 85) (1232, 4) (547, 4)
df_star: ['TCGA-D8-A146-01A', 'TCGA-AQ-A0Y5-01A', 'TCGA-C8-A274-01A', 'TCGA-BH-A0BD-01A', 'TCGA-B6-A1KC-01B', 'TCGA-AC-A62V-01A', 'TCGA-AO-A0J5-01A', 'TCGA-BH-A0B1-01A', 'TCGA-A2-A0YM-01A', 'TCGA-AO-A03N-01B', 'TCGA-AO-A1KQ-01A', 'TCGA-E2-A1LI-01A', 'TCGA-BH-A18L-11A', 'TCGA-B6-A0WV-01A', 'TCGA-E2-A1LE-01A', 'TCGA-A2-A0CO-01A', 'TCGA-AN-A0AK-01A', 'TCGA-BH-A28Q-01A', 'TCGA-3C-AALJ-01A', 'TCGA-BH-A0BM-01A', 'TCGA-AO-A03O-01A', 'TCGA-AC-A4ZE-01A', 'TCGA-E9-A22A-01A', 'TCGA-AR-A0TW-01A', 'TCGA-BH-A1FE-01A', 'TCGA-BH-A1FE-11B', 'TCGA-A2-A3XX-01A', 'TCGA-A8-A095-01A', 'TCGA-AN-A0FL-01A', 'TCGA-D8-A1Y2-01A', 'TCGA-C8-A138-01A', 'TCGA-AR-A255-01A', 'TCGA-GM-A5PV-01A', 'TCGA-E2-A108-01A', 'TCGA-AC-A2BM-01A', 'TCGA-E2-A1IN-01A', 'TCGA-A8-A08I-01A', 'TCGA-AC-A3W7-01A', 'TCGA-E9-A1QZ-01A', 'TCGA-EW-A1PH-01A', 'TCGA-BH-A0W7-01A', 'TCGA-AR-A24K-01A', 'TCGA-BH-A0H9-01A', 'TCGA-BH-A209-11A', 'TCGA-A8-A09B-01A', 'TCGA-A2-A0YT-01A', 'TCGA-BH-A0B5-01A', 'TCGA-

In [97]:
df_pam50['PAM50'].value_counts()

PAM50
LumA      232
LumB      129
Basal      98
Her2       58
Normal     30
Name: count, dtype: int64

To get our data ready to merge, we need to standardize the IDs of the samples. TCGA has a stardized barcode system

TCGA - A1 - A0SK - 01A - 11R - A084 - 07
  │    │     │      │     │     │     │
  │    │     │      │     │     │     └─ sequencing center
  │    │     │      │     │     └─ plate ID
  │    │     │      │     └─ portion + analyte type
  │    │     │      └─ sample type (01A = primary tumor, 11A = normal)
  │    │     └─ participant ID
  │    └─ tissue source site
  └─ project

Most of our data files have already truncated at sample type, so we will do the same to the PAM50 IDs so they match.

In [98]:
# Keep only the first 16 chars of the sample ID
df_pam50['Sample'] = df_pam50['Sample'].str[:16]

# Verify truncation did not produce any identical sample names
print('Any duplicated? ', df_pam50['Sample'].duplicated().any())

# Verify sample IDs match between datasets
print(len(pd.Index(df_pam50['Sample']).intersection(df_star.columns)))
print(len(pd.Index(df_pam50['Sample']).intersection(df_clinical['sample'])))
print(len(pd.Index(df_pam50['Sample']).intersection(df_survival['sample'])))

# Common sample IDs across datasets
print(len(pd.Index(df_pam50['Sample']).intersection(df_clinical['sample']).intersection(df_star.columns).intersection(df_survival['sample'])))

# For my initial analysis I will just be using the clinical, star, and pam50 dfs
common_samples = pd.Index(df_pam50['Sample']).intersection(df_clinical['sample']).intersection(df_star.columns)
print(common_samples)

Any duplicated?  False
546
547
534
533
Index(['TCGA-AN-A0FL-01A', 'TCGA-A1-A0SK-01A', 'TCGA-BH-A0HL-01A',
       'TCGA-BH-A0HN-01A', 'TCGA-BH-A0E0-01A', 'TCGA-AN-A0AL-01A',
       'TCGA-A1-A0SO-01A', 'TCGA-AN-A0G0-01A', 'TCGA-A8-A08X-01A',
       'TCGA-E2-A159-01A',
       ...
       'TCGA-AN-A0XV-01A', 'TCGA-BH-A0HK-01A', 'TCGA-E2-A14Q-01A',
       'TCGA-BH-A0BD-01A', 'TCGA-AR-A1AP-01A', 'TCGA-E2-A14Z-01A',
       'TCGA-AR-A1AX-01A', 'TCGA-BH-A0BS-01A', 'TCGA-A2-A0CY-01A',
       'TCGA-B6-A0RS-01A'],
      dtype='object', length=546)


# Merging and Saving Dataframes

For analysis we want to condense the dataframes into the expression data and the metadata. To do this, we can leave the df_star alone (but we will rename it df_expression), and we will combine the clinical and pam50 dfs into df_metadata.

In [99]:
# Filter our dataframes so only the common samples remain
df_clinical_filtered = df_clinical[df_clinical['sample'].isin(common_samples)]
df_pam50_filtered = df_pam50[df_pam50['Sample'].isin(common_samples)]
df_star_filtered = df_star[common_samples]

# Sanity check
print(df_clinical_filtered.shape)
print(df_pam50_filtered.shape)
print(df_star_filtered.shape)

(546, 85)
(546, 4)
(60660, 546)


In [100]:
# Merge clinical + pam50 dfs into metadata df
df_metadata = pd.merge(df_clinical_filtered, df_pam50_filtered, left_on='sample', right_on='Sample', how='inner')
print(df_metadata.shape)

# Rename df_star
df_expression = df_star_filtered
print(df_expression.shape)

(546, 89)
(60660, 546)


In [101]:
# Organize data
df_metadata = df_metadata.set_index('sample')
df_expression = df_expression.T

In [102]:
# Save dfs as csv files for safe keeping (and importing into other notebooks)
df_metadata.to_csv('/Users/nick/projects/bioinformatics-project/data/processed/metadata.csv')
df_expression.to_csv('/Users/nick/projects/bioinformatics-project/data/processed/expression.csv')